In [ ]:
import logging
import sys
import datetime

In [2]:
figures = "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/figures"
obj = "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects"

# CellphoneDB 

## Download database from source

### Display database versions

In [3]:
from IPython.display import HTML, display
from cellphonedb.utils import db_releases_utils
import pandas as pd
import glob
import os

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
display(HTML(db_releases_utils.get_remote_database_versions_html()['db_releases_html_table']))

### Define the version and the path to download database

In [6]:
# -- Version of the databse
cpdb_version = 'v5.0.0'

# -- Path where the input files to generate the database are located
cpdb_target_dir = os.path.join(f'{obj}', 'cellphoneDB_db',cpdb_version)

In [7]:
cpdb_target_dir

'/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/cellphoneDB_db/v5.0.0'

### Download database

In [ ]:
from cellphonedb.utils import db_utils

db_utils.download_database(cpdb_target_dir, cpdb_version)

## How to run CellphoneDB for the statistical method

### Input files

In [8]:
import scanpy as sc
import numpy as np
import pandas as pd
from anndata import AnnData
import pathlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import sys

In [9]:
from matplotlib.pyplot import MultipleLocator

In [10]:
# load the samples_all_annotation_diseased.h5ad with count layer
samples_all_diseased = sc.read_h5ad(f'{obj}/samples_all_annotation_diseased.h5ad')

# load the samples_all_annotation_healthy.h5ad with count layer
samples_all_healthy = sc.read_h5ad(f'{obj}/samples_all_annotation_healthy.h5ad')


In [11]:
# get the normalised_log_counts
sc.pp.normalize_total(samples_all_diseased)
sc.pp.log1p(samples_all_diseased)

sc.pp.normalize_total(samples_all_healthy)
sc.pp.log1p(samples_all_healthy)

In [12]:
# split the T_B merged adata into young and old objects
T_B_clusters = ['CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                'IFITM3_CD8_Teffector', 'GZMB_CD8_Teffector', 'GZMK_CD8_Tem', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex', 
                'Naive_B', 'Memory_B']

# diseased
samples_all_diseased_T_B = samples_all_diseased[samples_all_diseased.obs['sub_anno'].isin(T_B_clusters)].copy()
samples_all_diseased_T_B_young = samples_all_diseased_T_B[samples_all_diseased_T_B.obs['Age_type'] == 'Young'].copy()
samples_all_diseased_T_B_old = samples_all_diseased_T_B[samples_all_diseased_T_B.obs['Age_type'] == 'Old'].copy()

#healthy
samples_all_healthy_T_B = samples_all_healthy[samples_all_healthy.obs['sub_anno'].isin(T_B_clusters)].copy()
samples_all_healthy_T_B_young = samples_all_healthy_T_B[samples_all_healthy_T_B.obs['Age_type'] == 'Young'].copy()
samples_all_healthy_T_B_old = samples_all_healthy_T_B[samples_all_healthy_T_B.obs['Age_type'] == 'Old'].copy()


In [13]:
AnnData.write_h5ad(samples_all_diseased_T_B_young, filename=f'{obj}/CellphoneDB_db/samples_all_diseased_T_B_young_normalised_log_counts.h5ad')
AnnData.write_h5ad(samples_all_diseased_T_B_old, filename=f'{obj}/CellphoneDB_db/samples_all_diseased_T_B_old_normalised_log_counts.h5ad')

AnnData.write_h5ad(samples_all_healthy_T_B_young, filename=f'{obj}/CellphoneDB_db/samples_all_healthy_T_B_young_normalised_log_counts.h5ad')
AnnData.write_h5ad(samples_all_healthy_T_B_old, filename=f'{obj}/CellphoneDB_db/samples_all_healthy_T_B_old_normalised_log_counts.h5ad')

In [14]:
# get the meta_file (sub_anno)

#diseased
samples_all_diseased_T_B_young_subAnno_meta_file = samples_all_diseased_T_B_young.obs['Age_type'].str.cat(samples_all_diseased_T_B_young.obs['sub_anno'], sep='_')
samples_all_diseased_T_B_young_subAnno_meta_file = samples_all_diseased_T_B_young_subAnno_meta_file.to_frame().rename_axis('barcode_sample').reset_index().rename(columns = {'Age_type' : 'cell_type'})

samples_all_diseased_T_B_old_subAnno_meta_file = samples_all_diseased_T_B_old.obs['Age_type'].str.cat(samples_all_diseased_T_B_old.obs['sub_anno'], sep='_')
samples_all_diseased_T_B_old_subAnno_meta_file = samples_all_diseased_T_B_old_subAnno_meta_file.to_frame().rename_axis('barcode_sample').reset_index().rename(columns = {'Age_type' : 'cell_type'})


#healthy
samples_all_healthy_T_B_young_subAnno_meta_file = samples_all_healthy_T_B_young.obs['Age_type'].str.cat(samples_all_healthy_T_B_young.obs['sub_anno'], sep='_')
samples_all_healthy_T_B_young_subAnno_meta_file = samples_all_healthy_T_B_young_subAnno_meta_file.to_frame().rename_axis('barcode_sample').reset_index().rename(columns = {'Age_type' : 'cell_type'})

samples_all_healthy_T_B_old_subAnno_meta_file = samples_all_healthy_T_B_old.obs['Age_type'].str.cat(samples_all_healthy_T_B_old.obs['sub_anno'], sep='_')
samples_all_healthy_T_B_old_subAnno_meta_file = samples_all_healthy_T_B_old_subAnno_meta_file.to_frame().rename_axis('barcode_sample').reset_index().rename(columns = {'Age_type' : 'cell_type'})

In [15]:
# Check barcodes in metadata and counts are the same
print(
list(samples_all_diseased_T_B_young.obs.index).sort() == list(samples_all_diseased_T_B_young_subAnno_meta_file['barcode_sample']).sort(),
list(samples_all_diseased_T_B_old.obs.index).sort() == list(samples_all_diseased_T_B_old_subAnno_meta_file['barcode_sample']).sort())

True True


In [16]:
# Check barcodes in metadata and counts are the same
print(
list(samples_all_healthy_T_B_young.obs.index).sort() == list(samples_all_healthy_T_B_young_subAnno_meta_file['barcode_sample']).sort(),
list(samples_all_healthy_T_B_old.obs.index).sort() == list(samples_all_healthy_T_B_old_subAnno_meta_file['barcode_sample']).sort())

True True


In [17]:
#save meta files

##subIdentity diseased
samples_all_diseased_T_B_young_subAnno_meta_file.to_csv(f'{obj}/CellphoneDB_db/samples_all_diseased_T_B_young_subAnno_meta_file.tsv', sep = '\t', index=False)
samples_all_diseased_T_B_old_subAnno_meta_file.to_csv(f'{obj}/CellphoneDB_db/samples_all_diseased_T_B_old_subAnno_meta_file.tsv', sep = '\t', index=False)

##subIdentity healthy
samples_all_healthy_T_B_young_subAnno_meta_file.to_csv(f'{obj}/CellphoneDB_db/samples_all_healthy_T_B_young_subAnno_meta_file.tsv', sep = '\t', index=False)
samples_all_healthy_T_B_old_subAnno_meta_file.to_csv(f'{obj}/CellphoneDB_db/samples_all_healthy_T_B_old_subAnno_meta_file.tsv', sep = '\t', index=False)




In [18]:
cpdb_file_path = f'{obj}/CellphoneDB_db/v5.0.0/cellphonedb.zip'

#diseased
diseased_T_B_young_subAnno_meta_file_path = f'{obj}/CellphoneDB_db/samples_all_diseased_T_B_young_subAnno_meta_file.tsv'
diseased_T_B_young_degs_file_path = f'{obj}/CellphoneDB_db/diseased_T_B_young_subAnno_DEGs_file.tsv'
diseased_T_B_young_counts_file_path = f'{obj}/CellphoneDB_db/samples_all_diseased_T_B_young_normalised_log_counts.h5ad'

diseased_T_B_old_subAnno_meta_file_path = f'{obj}/CellphoneDB_db/samples_all_diseased_T_B_old_subAnno_meta_file.tsv'
diseased_T_B_old_degs_file_path = f'{obj}/CellphoneDB_db/diseased_T_B_old_subAnno_DEGs_file.tsv'
diseased_T_B_old_counts_file_path = f'{obj}/CellphoneDB_db/samples_all_diseased_T_B_old_normalised_log_counts.h5ad'

#healthy
healthy_T_B_young_subAnno_meta_file_path = f'{obj}/CellphoneDB_db/samples_all_healthy_T_B_young_subAnno_meta_file.tsv'
healthy_T_B_young_degs_file_path = f'{obj}/CellphoneDB_db/healthy_T_B_young_subAnno_DEGs_file.tsv'
healthy_T_B_young_counts_file_path = f'{obj}/CellphoneDB_db/samples_all_healthy_T_B_young_normalised_log_counts.h5ad'

healthy_T_B_old_subAnno_meta_file_path = f'{obj}/CellphoneDB_db/samples_all_healthy_T_B_old_subAnno_meta_file.tsv'
healthy_T_B_old_degs_file_path = f'{obj}/CellphoneDB_db/healthy_T_B_old_subAnno_DEGs_file.tsv'
healthy_T_B_old_counts_file_path = f'{obj}/CellphoneDB_db/samples_all_healthy_T_B_old_normalised_log_counts.h5ad'


### DEGs analysis (Method 3)

In [19]:
from cellphonedb.src.core.methods import cpdb_degs_analysis_method

In [20]:
#method3;diseased;young;T_B_subAnno
cpdb_diseased_T_B_young_results_deg_analysis = cpdb_degs_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = diseased_T_B_young_subAnno_meta_file_path,      # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = diseased_T_B_young_counts_file_path,             # mandatory: normalized count matrix.
    degs_file_path = diseased_T_B_young_degs_file_path, 
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                 # optional: whether to score interactions or not.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 30,                                     # number of threads to use in the analysis.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = f'{obj}/CellphoneDB_db',                          # Path to save results.
    output_suffix = 'samples_all_diseased_T_B_young_subAnno_cpdb_method3'                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

[ ][CORE][29/08/26-13:38:20][INFO] [Cluster DEGs Analysis] Threshold:0.05 Precision:3
Reading user files...
The following user files were loaded successfully:
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/samples_all_diseased_T_B_young_normalised_log_counts.h5ad
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/samples_all_diseased_T_B_young_subAnno_meta_file.tsv
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/diseased_T_B_young_subAnno_DEGs_file.tsv
[ ][CORE][29/08/26-13:38:26][INFO] Running Real Analysis
[ ][CORE][29/08/26-13:38:26][INFO] Running DEGs-based Analysis
[ ][CORE][29/08/26-13:38:26][INFO] Building results
[ ][CORE][29/08/26-13:38:26][INFO] Scoring interactions: Filtering genes per cell type..


100%|██████████| 11/11 [00:00<00:00, 66.89it/s]

[ ][CORE][29/08/26-13:38:27][INFO] Scoring interactions: Calculating mean expression of each gene per group/cell type..



100%|██████████| 11/11 [00:00<00:00, 248.05it/s]
/public/home/liwang/anaconda3/envs/jupyter-ai/lib/python3.11/site-packages/cellphonedb/utils/scoring_utils.py:138: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  matrix[index_name].replace(to_replace=id2name, inplace=True)


[ ][CORE][29/08/26-13:38:27][INFO] Scoring interactions: Calculating scores for all interactions and cell types..


100%|██████████| 121/121 [00:03<00:00, 30.74it/s]


Saved deconvoluted to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_deconvoluted_samples_all_diseased_T_B_young_subAnno_cpdb_method3.txt
Saved deconvoluted_percents to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_deconvoluted_percents_samples_all_diseased_T_B_young_subAnno_cpdb_method3.txt
Saved means to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_means_samples_all_diseased_T_B_young_subAnno_cpdb_method3.txt
Saved relevant_interactions to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_relevant_interactions_samples_all_diseased_T_B_young_subAnno_cpdb_method3.txt
Saved significant_means to /public

In [21]:
#method3;diseased;old;T_B_subAnno
cpdb_diseased_T_B_old_results_deg_analysis = cpdb_degs_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = diseased_T_B_old_subAnno_meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = diseased_T_B_old_counts_file_path,             # mandatory: normalized count matrix.
    degs_file_path = diseased_T_B_old_degs_file_path, 
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                 # optional: whether to score interactions or not.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 30,                                     # number of threads to use in the analysis.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = f'{obj}/CellphoneDB_db',                          # Path to save results.
    output_suffix = 'samples_all_diseased_T_B_old_subAnno_cpdb_method3'                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

[ ][CORE][29/08/26-13:38:36][INFO] [Cluster DEGs Analysis] Threshold:0.05 Precision:3
Reading user files...
The following user files were loaded successfully:
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/samples_all_diseased_T_B_old_normalised_log_counts.h5ad
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/samples_all_diseased_T_B_old_subAnno_meta_file.tsv
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/diseased_T_B_old_subAnno_DEGs_file.tsv
[ ][CORE][29/08/26-13:38:45][INFO] Running Real Analysis
[ ][CORE][29/08/26-13:38:45][INFO] Running DEGs-based Analysis
[ ][CORE][29/08/26-13:38:45][INFO] Building results
[ ][CORE][29/08/26-13:38:45][INFO] Scoring interactions: Filtering genes per cell type..


100%|██████████| 11/11 [00:00<00:00, 32.53it/s]

[ ][CORE][29/08/26-13:38:45][INFO] Scoring interactions: Calculating mean expression of each gene per group/cell type..



100%|██████████| 11/11 [00:00<00:00, 124.05it/s]
/public/home/liwang/anaconda3/envs/jupyter-ai/lib/python3.11/site-packages/cellphonedb/utils/scoring_utils.py:138: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  matrix[index_name].replace(to_replace=id2name, inplace=True)


[ ][CORE][29/08/26-13:38:46][INFO] Scoring interactions: Calculating scores for all interactions and cell types..


100%|██████████| 121/121 [00:03<00:00, 30.63it/s]


Saved deconvoluted to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_deconvoluted_samples_all_diseased_T_B_old_subAnno_cpdb_method3.txt
Saved deconvoluted_percents to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_deconvoluted_percents_samples_all_diseased_T_B_old_subAnno_cpdb_method3.txt
Saved means to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_means_samples_all_diseased_T_B_old_subAnno_cpdb_method3.txt
Saved relevant_interactions to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_relevant_interactions_samples_all_diseased_T_B_old_subAnno_cpdb_method3.txt
Saved significant_means to /public/home/li

In [22]:
#method3;healthy;young;T_B_subAnno
cpdb_healthy_T_B_young_results_deg_analysis = cpdb_degs_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = healthy_T_B_young_subAnno_meta_file_path,      # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = healthy_T_B_young_counts_file_path,             # mandatory: normalized count matrix.
    degs_file_path = healthy_T_B_young_degs_file_path, 
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                 # optional: whether to score interactions or not.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 30,                                     # number of threads to use in the analysis.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = f'{obj}/CellphoneDB_db',                          # Path to save results.
    output_suffix = 'samples_all_healthy_T_B_young_subAnno_cpdb_method3'                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

[ ][CORE][29/08/26-13:38:58][INFO] [Cluster DEGs Analysis] Threshold:0.05 Precision:3
Reading user files...
The following user files were loaded successfully:
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/samples_all_healthy_T_B_young_normalised_log_counts.h5ad
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/samples_all_healthy_T_B_young_subAnno_meta_file.tsv
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/healthy_T_B_young_subAnno_DEGs_file.tsv
[ ][CORE][29/08/26-13:39:00][INFO] Running Real Analysis
[ ][CORE][29/08/26-13:39:00][INFO] Running DEGs-based Analysis
[ ][CORE][29/08/26-13:39:00][INFO] Building results
[ ][CORE][29/08/26-13:39:00][INFO] Scoring interactions: Filtering genes per cell type..


100%|██████████| 11/11 [00:00<00:00, 159.24it/s]

[ ][CORE][29/08/26-13:39:00][INFO] Scoring interactions: Calculating mean expression of each gene per group/cell type..



100%|██████████| 11/11 [00:00<00:00, 528.24it/s]
/public/home/liwang/anaconda3/envs/jupyter-ai/lib/python3.11/site-packages/cellphonedb/utils/scoring_utils.py:138: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  matrix[index_name].replace(to_replace=id2name, inplace=True)


[ ][CORE][29/08/26-13:39:00][INFO] Scoring interactions: Calculating scores for all interactions and cell types..


100%|██████████| 121/121 [00:04<00:00, 28.87it/s]


Saved deconvoluted to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_deconvoluted_samples_all_healthy_T_B_young_subAnno_cpdb_method3.txt
Saved deconvoluted_percents to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_deconvoluted_percents_samples_all_healthy_T_B_young_subAnno_cpdb_method3.txt
Saved means to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_means_samples_all_healthy_T_B_young_subAnno_cpdb_method3.txt
Saved relevant_interactions to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_relevant_interactions_samples_all_healthy_T_B_young_subAnno_cpdb_method3.txt
Saved significant_means to /public/hom

In [23]:
#method3;healthy;old;T_B_subAnno
cpdb_healthy_T_B_old_results_deg_analysis = cpdb_degs_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = healthy_T_B_old_subAnno_meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = healthy_T_B_old_counts_file_path,             # mandatory: normalized count matrix.
    degs_file_path = healthy_T_B_old_degs_file_path, 
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                 # optional: whether to score interactions or not.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 30,                                     # number of threads to use in the analysis.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = f'{obj}/CellphoneDB_db',                          # Path to save results.
    output_suffix = 'samples_all_healthy_T_B_old_subAnno_cpdb_method3'                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

[ ][CORE][29/08/26-13:39:10][INFO] [Cluster DEGs Analysis] Threshold:0.05 Precision:3
Reading user files...
The following user files were loaded successfully:
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/samples_all_healthy_T_B_old_normalised_log_counts.h5ad
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/samples_all_healthy_T_B_old_subAnno_meta_file.tsv
/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/healthy_T_B_old_subAnno_DEGs_file.tsv
[ ][CORE][29/08/26-13:39:11][INFO] Running Real Analysis
[ ][CORE][29/08/26-13:39:11][INFO] Running DEGs-based Analysis
[ ][CORE][29/08/26-13:39:11][INFO] Building results
[ ][CORE][29/08/26-13:39:11][INFO] Scoring interactions: Filtering genes per cell type..


100%|██████████| 11/11 [00:00<00:00, 151.82it/s]

[ ][CORE][29/08/26-13:39:11][INFO] Scoring interactions: Calculating mean expression of each gene per group/cell type..



100%|██████████| 11/11 [00:00<00:00, 488.22it/s]
/public/home/liwang/anaconda3/envs/jupyter-ai/lib/python3.11/site-packages/cellphonedb/utils/scoring_utils.py:138: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  matrix[index_name].replace(to_replace=id2name, inplace=True)


[ ][CORE][29/08/26-13:39:12][INFO] Scoring interactions: Calculating scores for all interactions and cell types..


100%|██████████| 121/121 [00:03<00:00, 30.29it/s]


Saved deconvoluted to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_deconvoluted_samples_all_healthy_T_B_old_subAnno_cpdb_method3.txt
Saved deconvoluted_percents to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_deconvoluted_percents_samples_all_healthy_T_B_old_subAnno_cpdb_method3.txt
Saved means to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_means_samples_all_healthy_T_B_old_subAnno_cpdb_method3.txt
Saved relevant_interactions to /public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects/CellphoneDB_db/degs_analysis_relevant_interactions_samples_all_healthy_T_B_old_subAnno_cpdb_method3.txt
Saved significant_means to /public/home/liwang